## **Import libraries**

In [2]:
import subprocess
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import time
from sklearn.metrics import average_precision_score
import threading
from datetime import datetime
import random

In [47]:
# -Moises-
def fitness(chromosome: str) -> float:
    if not chromosome:
        return 0.0
    
    total_weight = 0
    for char in chromosome.lower():
        if 'a' <= char <= 'z':
            total_weight += ord(char) - ord('a') + 1
            
    max_possible_weight = 26 * len(chromosome)
    
    return total_weight / max_possible_weight

In [48]:
fitness("SKQ")

0.6025641025641025

## **Functions**

In [20]:
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

### **Initial Population**

In [ ]:
def generate_initial_population(peptide_length, population_size):
    population = []
    
    for _ in range(population_size):
        peptide = ''.join(random.choices(amino_acids, k=peptide_length))
        population.append(peptide)
        
    return population

In [14]:
# Example
generate_initial_population(20, 3)

['LPIVNAELMWAKCERQERPK', 'QFNVRNCGAFWLSRSGNPLE', 'WHCDNPPHDWLAQPRPQMDF']

### **Crossover**

In [34]:
import random

def uniform_crossover(parent1, parent2):
    child = []
    
    for p1_gene, p2_gene in zip(parent1, parent2):
        # Randomly generate a binary mask value (0 or 1)
        # to determine which parent's gene will be selected.
        mask = random.randint(0, 1)
        
        if mask == 0:
            child.append(p1_gene)
        else:
            child.append(p2_gene)
            
    return "".join(child)

In [32]:
# Example
uniform_crossover("MTASGKLRVPHA", "TIHWSGATGAGA")

'MIHWGGLTGPHA'

### **Mutation**

In [ ]:
def mutate_sequence(sequence, mutation_rate=0.8):
    mutated_sequence = []
    
    for gene in sequence:
        if random.random() < mutation_rate:
            possible_mutations = [aa for aa in amino_acids if aa != gene]
            mutated_sequence.append(random.choice(possible_mutations))
        else:
            mutated_sequence.append(gene)
            
    return "".join(mutated_sequence)

In [36]:
# Example
mutate_sequence("TISGHAW", mutation_rate=0.8)

'EWKAVKR'

### **Tournament**

In [59]:
def tournament(population, k=2):
    contenders = random.sample(population, k)
    #print(contenders)
    winner = max(contenders, key=lambda individual: individual['fitness'])
    return winner

In [46]:
# Example
population = [
    {"sequence": "EKTTNMF", "fitness": 0.92},
    {"sequence": "HKVMRAT", "fitness": 0.31},
    {"sequence": "MMQYQEY", "fitness": 0.45},
    {"sequence": "MICRHPM", "fitness": 0.99},
    {"sequence": "CQRRCEW", "fitness": 0.12},
    {"sequence": "NQTSCSL", "fitness": 0.92},
    {"sequence": "TKHNYCN", "fitness": 0.03},
    {"sequence": "RDVEKFA", "fitness": 0.23}
]

tournament(population, 2)


[{'sequence': 'MMQYQEY', 'fitness': 0.45}, {'sequence': 'CQRRCEW', 'fitness': 0.12}]


{'sequence': 'MMQYQEY', 'fitness': 0.45}

## **Genetic Algorithm**

In [ ]:
def genetic_algorithm(
    peptide_length,
    fitness_function,
    generations=300,
    children_count=30,
):
    
    # Generate an initial population 
    raw_population = generate_initial_population(peptide_length, children_count)

    # Evaluate fitness for each individual in the initial population
    population = [
        {'sequence': seq, 'fitness': fitness_function(seq)}
        for seq in raw_population
    ]

    # Evolutionary loop 
    for generation in range(generations):
        next_generation = []

        # Create a new generation of offspring
        for _ in range(children_count):

            # Select two parent individuals using tournament selection
            parent1 = tournament(population)
            parent2 = tournament(population)

            # Perform uniform crossover between selected parents
            child_sequence = uniform_crossover(
                parent1['sequence'],
                parent2['sequence']
            )

            # Apply mutation to introduce genetic variability
            child_sequence = mutate_sequence(child_sequence)

            # Evaluate fitness of the resulting offspring
            child_fitness = fitness_function(child_sequence)

            # Store the new individual in the next generation
            next_generation.append({
                'sequence': child_sequence,
                'fitness': child_fitness
            })

        # Replace the current population with the newly generated one
        population = next_generation

        # Identify and display the best individual of the current generation
        best_individual = max(population, key=lambda ind: ind['fitness'])
        print(
            f"Best Sequence: {best_individual['sequence']} | "
            f"Fitness: {best_individual['fitness']:.4f}"
        )

    # After all generations, return the best individual found
    best_overall = max(population, key=lambda ind: ind['fitness'])
    return best_overall

In [60]:
genetic_algorithm(
    peptide_length = 10,
    fitness_function = fitness,
    generations=300,
    children_count=30,
    )

Best Sequence: NVVEWWVEIW | Fitness: 0.6462
Best Sequence: ESKPPSTRVW | Fitness: 0.6500
Best Sequence: QVQSNFRYWN | Fitness: 0.6731
Best Sequence: SMTEVVQMRW | Fitness: 0.6615
Best Sequence: PREQPRRSSY | Fitness: 0.6577
Best Sequence: VLEQSYQSSQ | Fitness: 0.6615
Best Sequence: RTQDSYMSQH | Fitness: 0.6154
Best Sequence: TQQYTPRMYT | Fitness: 0.7346
Best Sequence: KWWVTGRMYR | Fitness: 0.6923
Best Sequence: LQVYQNFLTT | Fitness: 0.6346
Best Sequence: TWHYGITLYY | Fitness: 0.6692
Best Sequence: YNPSWSVPSH | Fitness: 0.6962
Best Sequence: RYTQDSTWQL | Fitness: 0.6731
Best Sequence: LKYTYYSIDM | Fitness: 0.6269
Best Sequence: WWSSKSRLTS | Fitness: 0.7038
Best Sequence: RQWMVQVVKT | Fitness: 0.7115
Best Sequence: VCWKWRTSQK | Fitness: 0.6423
Best Sequence: VRKWQLHVSY | Fitness: 0.6808
Best Sequence: WWWRAYEMWQ | Fitness: 0.6577
Best Sequence: GTYSWNMPVP | Fitness: 0.6731
Best Sequence: WTTRKVEMYV | Fitness: 0.6885
Best Sequence: YSRYYVKQWC | Fitness: 0.7231
Best Sequence: YPFHYQPPGT | Fitn

{'sequence': 'QRWGSRPQQD', 'fitness': 0.6}

# **Your turn!!**

In [ ]:
## Function Mutation with restrictions; amino acids with same polarity

In [ ]:
## Genetic with mu+lambda